# 🥉 Bronze — Yellow Taxi raw ingestion

**Purpose.** Ingest raw NYC Yellow Taxi parquet files into a Bronze Delta table. The Bronze layer preserves the source exactly as it is — no filtering, no transformations — and adds two audit columns so that any row can be traced back to its source file and ingestion time.

**Source.** NYC TLC Trip Record Data — https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

**Output.** Delta table `workspace.bronze.yellow_taxi`.

**Audit columns added:**
- `_ingested_at` — UTC timestamp the row was written to Bronze
- `_source_file` — the source parquet URL the row came from

In [0]:
# Configuration -----------------------------------------------------------------
# Which months to ingest. The TLC publishes one parquet file per month,
# named yellow_tripdata_YYYY-MM.parquet. We start with two months — large
# enough that Spark is doing real work, small enough that Free Edition's
# quotas are comfortable.
SOURCE_MONTHS = [
    "2024-01",
    "2024-02",
]

# Public URL pattern for the TLC parquet files.
SOURCE_URL_TEMPLATE = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/"
    "yellow_tripdata_{month}.parquet"
)

# Target catalog / schema / table for the Bronze layer.
TARGET_CATALOG = "workspace"
TARGET_SCHEMA = "bronze"
TARGET_TABLE = "yellow_taxi"
FULL_TABLE_NAME = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{TARGET_TABLE}"

print("Will ingest months:", SOURCE_MONTHS)
print("Target table:      ", FULL_TABLE_NAME)

In [0]:
# Create the target schema if it doesn't already exist. CREATE SCHEMA is
# idempotent — running it twice does nothing harmful.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}")

# Tell Spark to use this schema by default for unqualified table references.
spark.sql(f"USE {TARGET_CATALOG}.{TARGET_SCHEMA}")

print(f"✓ Schema {TARGET_CATALOG}.{TARGET_SCHEMA} ready.")

In [0]:
import os
import urllib.request

# Create a Volume in the target schema — this is the Free Edition-friendly way
# to store raw files that Spark can then read. Volumes live inside the catalog
# alongside tables, so they participate in Unity Catalog governance.
VOLUME_NAME = "raw_files"
FULL_VOLUME = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{VOLUME_NAME}"
VOLUME_MOUNT_PATH = f"/Volumes/{TARGET_CATALOG}/{TARGET_SCHEMA}/{VOLUME_NAME}"

spark.sql(f"CREATE VOLUME IF NOT EXISTS {FULL_VOLUME}")
print(f"✓ Volume ready: {FULL_VOLUME}")
print(f"  Mount path:   {VOLUME_MOUNT_PATH}")

# Download each month's parquet file into the Volume.
local_paths = []
for month in SOURCE_MONTHS:
    url = SOURCE_URL_TEMPLATE.format(month=month)
    target_path = f"{VOLUME_MOUNT_PATH}/yellow_tripdata_{month}.parquet"

    if os.path.exists(target_path):
        size_mb = os.path.getsize(target_path) / (1024 * 1024)
        print(f"  ✓ already in volume ({size_mb:.1f} MB): {target_path}")
    else:
        print(f"  ↓ downloading: {url}")
        urllib.request.urlretrieve(url, target_path)
        size_mb = os.path.getsize(target_path) / (1024 * 1024)
        print(f"    saved {size_mb:.1f} MB → {target_path}")

    local_paths.append(target_path)

print(f"\n✓ {len(local_paths)} parquet file(s) staged in {VOLUME_MOUNT_PATH}")

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

# Read the parquet files now staged in the Volume. Spark on serverless
# is allowed to read from /Volumes/... paths natively.
raw_df = spark.read.parquet(*local_paths)

print(f"Schema ({len(raw_df.columns)} columns):")
raw_df.printSchema()

print("\nRow count (full read — may take ~30s):")
row_count = raw_df.count()
print(f"  {row_count:,} rows across {len(SOURCE_MONTHS)} months")

In [0]:
from pyspark.sql.functions import current_timestamp, col

# Add the two Bronze-layer audit columns.
# - _ingested_at: when this row was loaded into Bronze (UTC timestamp).
# - _source_file: which source file the row came from. Unity Catalog
#                 exposes this via the special _metadata.file_path column,
#                 replacing the older input_file_name() function.
bronze_df = (
    raw_df
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

print(f"Final schema ({len(bronze_df.columns)} columns — 2 audit columns added):")
bronze_df.printSchema()

# Show a tiny sample so we can sanity-check.
display(bronze_df.limit(3))

In [0]:
# Write the DataFrame to a Delta table. Overwrite mode is used here because
# Bronze ingestion is treated as idempotent — each run replaces the table.
# In a streaming / incremental scenario we would use append + MERGE instead.
(
    bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FULL_TABLE_NAME)
)

print(f"✓ Wrote {bronze_df.count():,} rows to {FULL_TABLE_NAME}")

In [0]:
# Read back from the Delta table to confirm it's queryable.
verify_df = spark.table(FULL_TABLE_NAME)
print(f"Bronze table row count: {verify_df.count():,}")

# Show the audit columns are populated correctly.
print("\nSample rows:")
display(
    verify_df
    .select(
        "VendorID",
        "tpep_pickup_datetime",
        "passenger_count",
        "fare_amount",
        "total_amount",
        "_ingested_at",
        "_source_file",
    )
    .limit(5)
)

# Quick sanity check: rows per source file.
print("\nRow count per source file:")
display(
    verify_df
    .groupBy("_source_file")
    .count()
    .orderBy("_source_file")
)